In [53]:
import numpy as np

from geevo.nodes import *
from geevo.graph import Graph2, Graph3
from tqdm.notebook import tqdm
from geevo.simulation import Simulator
from geevo.evolution.balancer import BalancerV2, Balancer

%load_ext autoreload
%autoreload 2

%matplotlib inline

def init_nmmo_graph(weights=None):
    conf = {
        Source: 2,  # 0, 1
        FixedPoolLimit: 4,  # 2, 3, 4, 5
        Converter: 3,  # 6, 7, 8
        RandomGate: 2,  # 9, 10
        Drain: 4,  # 11, 12, 13, 14
        Pool: 2  # 15, 16
    }
    # pool-2: health
    # pool-3: water
    # pool-4: count buffer food
    # pool-5: food counter --> not implemented
    # pool-6: food

    edge_list = [(0, 9), (0, 10), (1, 2),  # sources
                 (2, 11), (3, 12), (4, 8), (4, 14), (5, 13),  # pools1
                 (9, 7), (9, 15), (10, 6), (10, 16),  # random gates
                 (6, 3), (7, 4), (8, 5)  # converters
                 ]

    s_in_food = StateConnectionPoolRegister(variable_name="food", output_pool_id=5, register_input_id=0)
    s_in_water = StateConnectionPoolRegister(variable_name="water", output_pool_id=3, register_input_id=0)
    s_out_health_regain = StateConnectionRegisterEdge(output_register_id=0, edge_input_id=(1, 2), modifier=10)
    regain_health_register = Register(condition="water > 0 and food > 0", name="Regain Health Register")
    regain_health_register.add_input(s_in_food)
    regain_health_register.add_input(s_in_water)
    regain_health_register.add_output(s_out_health_regain)

    s_in_food = StateConnectionPoolRegister(variable_name="food", output_pool_id=5, register_input_id=0)
    s_in_water = StateConnectionPoolRegister(variable_name="water", output_pool_id=3, register_input_id=0)
    s_out_health_drop = StateConnectionRegisterEdge(output_register_id=0, edge_input_id=(2, 11), modifier=10)
    lose_health_register = Register(condition="water == 0 or food == 0", name="Lose Health Register")
    lose_health_register.add_input(s_in_food)
    lose_health_register.add_input(s_in_water)
    lose_health_register.add_output(s_out_health_drop)

    # end condition

    if weights is None:
        weights = [1, 1, 0,  # sources
                   0, 10, 1, 1, 10,  # pools  
                   0.06, 0.94, 0.74, 0.26,  # random gates
                   100, 2, 100]  # converters

    g3 = Graph3(config=conf, edge_list=edge_list, weights=weights,
                registers=[regain_health_register, lose_health_register])
    # Sources
    g3.nodes[0].name = "Step"
    g3.nodes[1].name = "Health"

    # Pools
    g3.nodes[2].name = "Health"
    g3.nodes[2].pool = 100
    g3.nodes[3].name = "Water"
    g3.nodes[3].pool = 100
    g3.nodes[4].name = "Food Count Buffer"
    g3.nodes[5].name = "Food"
    g3.nodes[5].pool = 100
    g3.nodes[14].name = "Food Counter"

    # g3.plot(figsize=(15, 7.5))
    return g3


def simulate_economies(graph1, graph2):
    # 1
    win_cond = EndCondition(graph1.nodes[14], 14, "Food Counter >= 5", WinStates.PLAYER_WINS)
    loose_cond = EndCondition(graph1.nodes[2], 2, "Health <= 0", WinStates.PLAYER_LOST)
    res1, win_loose1, game_length1 = graph1.simulate(100, win_conditions=[win_cond, loose_cond])

    win_cond = EndCondition(graph2.nodes[14], 14, "Food Counter >= 5", WinStates.PLAYER_WINS)
    loose_cond = EndCondition(graph2.nodes[2], 2, "Health <= 0", WinStates.PLAYER_LOST)
    res2, win_loose2, game_length2 = graph2.simulate(100, win_conditions=[win_cond, loose_cond])

    # evaluate winner
    if game_length1 > game_length2:
        if res2["Food Counter"][-1] >= 5:
            # print(f"Winner player 2: Player 2 collected 5 food before player 1 (only {res1['Food Counter'][-1]}).")
            return [1]
        else:
            # print(
            #     f"Winner player 1: Player 1 survived longer ({game_length1}) than player 2 ({game_length2}, {res2['Food Counter'][-1]}).")
            return [0]
    elif game_length1 < game_length2:
        if res1["Food Counter"][-1] >= 5:
            # print(f"Winner player 1: Player 1 collected 5 food before player 2 (only {res2['Food Counter'][-1]}).")
            return [0]
        else:
            # print(
                # f"Winner player 2: Player 2 survived longer ({game_length2}) than player 1 ({game_length1}, {res1['Food Counter'][-1]}).")
            return [1]
    else:
        # print("else")
        return [0, 1]
    # return res1, win_loose1, game_length1


def get_win_rates(weights=None, n=50):
    winners = []
    for _ in range(n):
        weights_p1 = [1, 1, 0,  # sources
                      0, 10, 1, 1, 10,  # pools  
                      0.058, 0.942, 0.752, 0.248,  # random gates
                      100, 2, 100]  # converters
        
        if weights is not None:
            changable_weights = [4, 7, 12, 14]
            for idx, w in zip(changable_weights, weights_p1):
                weights_p1[idx] = w

        economy_p1 = init_nmmo_graph(weights_p1)
        weights_p2 = [1, 1, 0,  # sources
                      0, 10, 1, 1, 10,  # pools  
                      0.106, 0.894, 0.752, 0.248,  # random gates
                      100, 2, 100]  # converters
        economy_p2 = init_nmmo_graph(weights_p2)
        winner = simulate_economies(economy_p1, economy_p2)
        winners.extend(winner)
    return round(np.mean(winners), 3)


def get_fitness(weights, balancing=0.5):
    win_rate = get_win_rates(weights)
    return abs(1 - win_rate - balancing)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [68]:
class BalancerV3:
    def __init__(self, f_init_graph, f_fitness, initial_weights, pop_size=10, n_sim=10, n_sim_steps=100, alpha=0.01):
        self.f_init_graph = f_init_graph
        self.f_fitness = f_fitness
        self.initial_weights = initial_weights
        self.changable_weights = [4, 7, 12, 14]

        
        self.monitor = self.monitor = {"best": [], "avg": []}
        self.pop_size = pop_size
        self.population = []
        self.n_sim = n_sim
        self.n_sim_steps = n_sim_steps
        self.init_population()
        self.result = None
        self.threshold = 1 - alpha

    def init_ind(self):
        weights = []
        for idx in self.changable_weights:
            if np.random.random(1)[0] >= 0.5:
                weights.append(self.initial_weights[idx]*(np.random.random(1)[0]+0.6))
            else:
                weights.append(self.initial_weights[idx])
        return weights

    def init_population(self):
        for i in range(self.pop_size):
            self.population.append(self.init_ind())

    def get_ind_fitness(self, ind):
        return self.f_fitness(ind)
    
    def get_fitness(self, return_always=False):
        fitness = []
        for ind in self.population:
            fitness.append(self.get_ind_fitness(ind))

        # sort by fitness
        fitness_sorted = np.argsort(fitness)
        pop_ = np.array(self.population, dtype=object)
        fitness = np.array(fitness)
        print(fitness[fitness_sorted][::-1])
        pop_ = pop_[fitness_sorted][::-1][:self.pop_size]
        self.population = pop_.tolist()
        # print(sorted(fitness)[-1])
        self._monitor(fitness)
        if fitness.max() >= self.threshold:
            self.result = self.population[0]
            return fitness.max()
        if return_always is True:
            return fitness.max()

    def crossover(self):
        indices = list(range(len(self.population)))
        random.shuffle(indices)
        new = []
        for idx in range(len(indices))[::2]:
            one = self.population[indices[idx]]
            two = self.population[indices[idx + 1]]
            split_point = np.random.randint(max(len(one), 1))
            new.append([*one[:split_point], *two[split_point:]])
            new.append([*one[:split_point], *two[split_point:]])

            # cross probabilistic weights
            # mean = np.mean(one[1]) * 0.2
            # if np.random.randint(1) == 1:  # + or -
            #     new.append([[*one[0][:split_point], *two[0][split_point:]], (abs(np.array(one[1]) + mean)).tolist()])
            #     new.append([[*one[0][:split_point], *two[0][split_point:]], (abs(np.array(two[1]) + mean)).tolist()])
            # else:
            #     new.append([[*one[0][:split_point], *two[0][split_point:]], (abs(np.array(one[1]) - mean)).tolist()])
            #     new.append([[*one[0][:split_point], *two[0][split_point:]], (abs(np.array(two[1]) - mean)).tolist()])
        self.population.extend(new)

    def mutate(self):
        selection = np.random.randint(len(self.population))
        selection_weight = np.random.randint(len(self.population[0]))
        mutation = np.random.randint(8)
        if np.random.randint(1) == 0:
            self.population[selection][selection_weight] += mutation
        else:
            self.population[selection][selection_weight] -= mutation
            if self.population[selection][selection_weight] < 1:
                self.population[selection][selection_weight] = 1

    def run(self, steps=100):
        iterations = steps
        for i in range(steps):
            self.crossover()
            self.mutate()
            if i % 5 == 0:
                self.population.append(self.init_ind())
            fitness = self.get_fitness(return_always=True)
            
            print(f"Iteration {i+1}, fitness: {fitness}")
            print(self.population)
            if fitness == -0.5:
                print(f"Stopped after {i} iteration with a fitness of: {fitness}")
                iterations = i
                break
        if fitness is None:
            fitness = self.get_fitness(return_always=True)
        return fitness, iterations

    def _monitor(self, fitness):
        self.monitor["best"].append(fitness.max())
        self.monitor["avg"].append(fitness.mean())

    def plot_monitor(self):
        fig = plt.figure(figsize=(7, 5))
        for k, v in self.monitor.items():
            plt.plot(list(range(len(v))), v, label=k)
        plt.legend()

In [46]:
l = [1,2,3]
l[::-1]

[3, 2, 1]

In [69]:
weights_p1 = [1, 1, 0,  # sources
              0, 10, 1, 1, 10,  # pools  
              0.058, 0.942, 0.752, 0.248,  # random gates
              100, 2, 100]  #
balancer = BalancerV3(init_nmmo_graph, f_fitness=get_fitness, initial_weights=weights_p1)
balancer.population

[[10, 8.170757866617677, 100, 100],
 [10, 7.8356252976438165, 100, 100],
 [10.2839552404261, 11.75471366918947, 69.58586467699077, 128.59039265839965],
 [10, 10, 113.86069578593738, 113.29866562363904],
 [6.530930138582068, 10, 100, 148.8977102303966],
 [11.043112788179371, 10, 121.5254350958868, 100],
 [10, 11.702224494272851, 119.20718792623994, 100],
 [10, 13.876897062897735, 100, 100],
 [8.619461879829819, 10.027926199078694, 140.3055626728659, 100],
 [13.94210026932945, 9.341665878490826, 87.98476826757071, 100]]

In [70]:
balancer.run(100)

[0.1   0.08  0.08  0.08  0.08  0.08  0.069 0.06  0.06  0.06  0.06  0.06
 0.04  0.04  0.04  0.029 0.02  0.02  0.01  0.    0.   ]
Iteration 1, fitness: 0.09999999999999998
[[13.94210026932945, 9.341665878490826, 87.98476826757071, 100], [10, 13.770466195877473, 90.07789888184097, 100], [10, 10, 113.86069578593738, 100], [10.2839552404261, 11.75471366918947, 69.58586467699077, 128.59039265839965], [10, 13.876897062897735, 100, 100], [10.2839552404261, 10.027926199078694, 140.3055626728659, 100], [10, 10, 113.86069578593738, 113.29866562363904], [10, 8.170757866617677, 87.98476826757071, 100], [6.530930138582068, 10, 100, 100], [11.043112788179371, 10, 121.5254350958868, 100]]
[0.16  0.12  0.1   0.088 0.08  0.08  0.069 0.06  0.06  0.06  0.04  0.04
 0.02  0.02  0.02  0.02  0.02  0.02  0.    0.   ]
Iteration 2, fitness: 0.15999999999999992
[[10, 8.170757866617677, 87.98476826757071, 100], [10, 13.876897062897735, 100, 100], [6.530930138582068, 10, 100, 100], [10, 10, 120.86069578593738, 100]

(0.12, 100)

In [67]:
balancer.population

[[11.239711499675629, 10, 100, 98.48440340890333],
 [16, 10, 100, 98.48440340890333],
 [11.239711499675629, 11, 100, 98.48440340890333],
 [10, 10, 100, 98.48440340890333],
 [11.239711499675629, 11, 100, 98.48440340890333],
 [16, 10, 100, 98.48440340890333],
 [16, 10, 100, 98.48440340890333],
 [11.239711499675629, 11, 100, 98.48440340890333],
 [11.239711499675629, 10, 100, 98.48440340890333],
 [10, 10, 100, 98.48440340890333]]